In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Scaling Up

Every experiment in this chapter so far chose its learning rate with a
sweep: a few dozen short runs, each lasting seconds to minutes on one GPU. A
model with billions of parameters, trained for months on thousands of
accelerators, allows no such sweep. Even a single extra full run may be
too expensive. Large-scale training therefore tunes a small proxy model and
reuses the hyperparameters it finds for the large one. If they do not carry
over, the large run diverges or trains slowly, and the error shows only after
much of the budget is spent.

This section asks when such a transfer works. We train a family of networks
that differ only in width, watch the best learning rate move as the network
widens, and trace the move to the size of the very first update. We then
change how each layer's learning rate and output scale depend on the width,
so that a learning rate tuned on the narrowest network stays good for the
widest. A diagnostic that takes seconds checks the change, a sweep tests the
transfer, and a last step connects the rules to the update sizes of Muon
(that section).

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

In [ ]:
import random

## Learning Rate and Width

### A Family of Widths

We want the smallest experiment in which a model can be scaled up while
remaining the same model. A multilayer perceptron on Fashion-MNIST serves:
three hidden layers of width $n$, with $n$ from 128 to 1,024. Every member
of the family computes the same kind of function; only the width differs.
Note where the width lives. The input matrix always has fan-in 784 and the
output matrix always has fan-out 10, so only the two $n \times n$ matrices
in the middle grow in both dimensions. This distinction matters below.
Because we will train several dozen of these models for a few hundred steps
each, we keep the whole training set on the accelerator as one tensor. The
60,000 images occupy less than 200 MB, and indexing them directly avoids
the overhead of a data loader in such short runs.

In [2]:
fashion = d2l.FashionMNIST()
device = d2l.try_gpu()
X = (fashion.train.data.float().reshape(-1, 784) / 255).to(device)
Y = fashion.train.targets.to(device)

We specify the initialization explicitly, because the initialization is
part of what a rule for scaling must prescribe: every weight is Gaussian
with variance $1/\textrm{fan-in}$, and every bias is zero. Together with one
learning rate shared by all layers, this is the *standard parametrization*
(SP). Up to constants it is what default initializers do and what the
Xavier and He schemes of that section prescribe. The
optimizer is created by a method, `configure_adam`, so that a variant of the
class can override it later.

In [3]:
class MLP(nn.Module):
    """A three-hidden-layer ReLU network under standard parametrization."""
    def __init__(self, width):
        super().__init__()
        self.fc_in = nn.Linear(784, width)
        self.fc_h1 = nn.Linear(width, width)
        self.fc_h2 = nn.Linear(width, width)
        self.fc_out = nn.Linear(width, 10)
        for lin in (self.fc_in, self.fc_h1, self.fc_h2, self.fc_out):
            nn.init.normal_(lin.weight, std=lin.in_features ** -0.5)
            nn.init.zeros_(lin.bias)

    def features(self, X):
        h = F.relu(self.fc_h1(F.relu(self.fc_in(X))))
        return F.relu(self.fc_h2(h))

    def forward(self, X):
        return self.fc_out(self.features(X))

    def configure_adam(self, lr):
        return torch.optim.Adam(self.parameters(), lr)

Training is a step-counted loop in the style of that section: 400
steps of Adam at batch size 512, about three passes over the data, at a
constant learning rate. A seed fixes both the initialization and the
sequence of minibatches, so runs with the same seed differ only in width,
parametrization, and learning rate. The score of a run is its cross-entropy
on the whole training set after the last step. A diverged run is capped at
2.5, slightly above the loss $\log 10 \approx 2.3$ of uniform guessing.

In [ ]:
def train_step(model, optimizer, Xb, Yb):
    loss = F.cross_entropy(model(Xb), Yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.detach()

def train_mlp(arch, width, lr, seed, num_steps=400, batch_size=512):
    torch.manual_seed(seed)
    model = arch(width).to(device)
    optimizer = model.configure_adam(lr)
    idx = np.random.default_rng(seed).integers(0, X.shape[0],
                                               (num_steps, batch_size))
    for i in idx:
        train_step(model, optimizer, X[i], Y[i])
    with torch.inference_mode():
        v = float(F.cross_entropy(model(X), Y))
    return min(v, 2.5) if math.isfinite(v) else 2.5

### Learning-Rate Sweep

We evaluate four widths and nine learning rates spaced by factors of two,
one short run per configuration, all from one random seed. The 36 runs stand
in for the proxy sweep of a large project. They show whether the learning
rate selected at one width serves the others.

In [ ]:
widths = [128, 256, 512, 1024]
lrs = [2 ** k for k in range(-13, -4)]
seed = random.getrandbits(32)  # One draw for every run of this section
sp_loss = {w: [train_mlp(MLP, w, lr, seed) for lr in lrs] for w in widths}
for w in widths:
    print(f'width {w:4d}: best lr {min(zip(sp_loss[w], lrs))[1]:.1e}')
d2l.plot(lrs, [sp_loss[w] for w in widths], 'learning rate',
         'training loss', xscale='log', ylim=[0.25, 0.8],
         legend=[f'width {w}' for w in widths])

Each width traces a U: too small a learning rate undertrains, and too large
a one destabilizes. On the right the wider networks tend to climb fastest,
in some runs off the top of the chart, where the capped runs sit. The U's do
not line up. The minimum slides left as the network widens: the best
learning rate at width 1,024 is several times smaller than at width
128, and the drift
continues at larger widths [@Yang.Hu.Babuschkin.ea.2022]. After tuning,
the wider networks match or beat the narrower ones, so width helps. But the
best region does not transfer. The width-128 optimum puts the width-1,024
network on the unstable side of its U, and the width-1,024 optimum
undertrains the narrow network. Under standard parametrization the right
learning rate depends on the width as well as on the task.

### The First Adam Step

The size of the first update explains the drift and shows how to correct
it. Consider a hidden weight matrix $\mathbf{W} \in \mathbb{R}^{n \times n}$
that receives activations $\mathbf{h}$ whose entries have typical size
$\overline{|h|}$. For a single example its gradient is an outer product,
$\mathbf{g} = \boldsymbol{\delta} \mathbf{h}^\top$, where
$\boldsymbol{\delta}$ is the error propagated back to the layer's output.
On the very first step, Adam's ratio $\hat{\mathbf{m}}/\sqrt{\hat{\mathbf{v}}}$
equals $\operatorname{sign}(\mathbf{g})$, because after bias correction the
first and second moment estimates are $\mathbf{g}$ and $\mathbf{g}^2$
(that section). The sign of an outer product factorizes,
$\operatorname{sign}(\delta_i h_j) = \operatorname{sign}(\delta_i)
\operatorname{sign}(h_j)$, so the update changes the layer's output by

$$
(\Delta\mathbf{W}\, \mathbf{h})_i
= -\eta \sum_{j=1}^{n} \operatorname{sign}(\delta_i h_j)\, h_j
= -\eta\, \operatorname{sign}(\delta_i) \sum_{j=1}^{n} |h_j|
\approx -\eta\, \operatorname{sign}(\delta_i)\, n\, \overline{|h|}.
$$

All $n$ terms of the sum have the same sign, because the update is built
from the very activations that it multiplies. Their sizes add up instead of
cancelling, and each output coordinate moves by about
$\eta\, n\, \overline{|h|}$. A minibatch gradient is a sum of such outer
products, and its sign no longer factorizes, so this argument holds only to
leading order. The coordinate check below confirms the width dependence by
measurement. Doubling the width doubles the change, so the largest stable
learning rate falls in proportion to $1/n$, the trend that the sweep shows.
The effect differs between layers. The input layer has fan-in 784 at every
width, and a bias has no fan-in at all, so their stable learning rates do
not move. The output matrix has fan-in $n$ as well, so its step grows like
those of the hidden matrices. It gets its own correction below. A single global $\eta$ is
a compromise between layers that scale differently, and the layers with
fan-in $n$ pull it down.

## Maximal Update Parametrization

### The Rules

If the stable learning rate of each layer scales differently with width,
one learning rate should not have to serve them all. The *maximal update
parametrization* (muP) [@Yang.Hu.Babuschkin.ea.2022] writes the width
dependence into per-layer rules. Relative to a chosen *base width*, each
layer's learning rate, and one scale factor in the forward pass, absorbs that
layer's own scaling. What remains is a base learning rate that does not
depend on the width and can be tuned on the smallest member of the family.
The rules are derived from the limit of infinite width in which
every layer's activations stay of order one *and* every layer keeps learning
features [@Yang.Hu.2021]. Larger updates make activations grow with
width, as in the equation; smaller ones freeze the
features in the limit. "Maximal update" names the largest updates that keep
the activations stable.

For Adam and a width multiplier $m = n / n_{\textrm{base}}$, the rules fit
in a table.

:The maximal update parametrization for Adam, relative to a chosen base
width. The hidden-matrix learning rate and the output-logit multiplier
depend on the width; the initialization, the learning rates of the input
weights and biases, and the rest of the forward pass do not.

| parameters | initialization | Adam learning rate | forward pass |
|:--|:--|:--|:--|
| input weights and all biases | unchanged | $\eta$ | unchanged |
| hidden matrices | variance $\propto 1/\textrm{fan-in}$ (unchanged) | $\eta / m$ | unchanged |
| output matrix | unchanged | $\eta$ | logits $\times\, 1/m$ |

The initialization column is the one that does *not* change: variance
$\propto 1/\textrm{fan-in}$ is right at every width, and `MLP` already uses
it. The hidden learning rate shrinks as $1/m$, which cancels the $n$-fold
sum of the equation. The output layer keeps its learning
rate, but its logits are divided by $m$, which cancels the same sum on the
output side while letting the head keep learning. Input weights and biases,
whose fan-in never grows, are left alone. For language models the token
embedding belongs to this class too, which is why muP treats embeddings
apart from hidden matrices. Transformers need one further rule: attention
logits scaled by $1/d$ rather than $1/\sqrt{d}$. SGD has its own column of
exponents, and several algebraically equivalent formulations are in use.
The practitioner's guide of @Dey.Anthony.Hestness.2024 lays out the
full table and its pitfalls. Here the whole change is a subclass:

In [ ]:
class MuMLP(MLP):
    """The same network under muP, relative to a width-1,024 base."""
    def __init__(self, width, base_width=1024):
        super().__init__(width)
        self.m = width / base_width

    def forward(self, X):
        # Rule 1: scale the output matrix's logits; the bias is untouched
        return self.fc_out(self.features(X) / self.m)

    def configure_adam(self, lr):
        hidden = [self.fc_h1.weight, self.fc_h2.weight]
        rest = [p for p in self.parameters()
                if not any(p is q for q in hidden)]
        return torch.optim.Adam([              # Rule 2: hidden LR / m
            {'params': rest, 'lr': lr},
            {'params': hidden, 'lr': lr / self.m}])

At the base width $m = 1$, and `MuMLP` is `MLP`. Which width serves as the
base is a free choice. Every choice gives the same dependence on the width;
choices differ only by constant factors per layer, such as how fast the input
layer learns relative to the hidden ones and how large the output multiplier
is. These constants are hyperparameters in their own right, and the
practitioner's guide tunes them on the proxy along with the learning rate. We
avoid those extra sweeps by putting the base at the widest member of the
family, 1,024, where `MuMLP` is the standard network with one learning rate
for all layers. The narrower members, including the width-128 model that we
tune, have $m < 1$. A base at width 128 would instead fix the constants where
they matter least: in our runs it leaves tuned muP at width 1,024 one to
three hundredths of a nat above tuned standard parametrization. An exercise
traces which constants make the difference.

### The Coordinate Check

The *coordinate check* tests the width scaling of
the equation directly, and the practitioner's guide of
@Dey.Anthony.Hestness.2024 recommends it as the first test of any
muP implementation. Under the rules used here, the hidden activations should
stay of order one after a training step at every width, while the logits
follow their explicit $1/m$ scaling. Activations that grow with width reveal
a scaling error. We therefore build the family across widths, apply one Adam
update at a learning rate in the middle of the grid, and record the mean
absolute activation of each layer. One step is the cleanest probe:
the equation describes it directly, while later steps
compound the growth and mix it with feedback from the loss.

In [ ]:
def activations(model, Xb):
    with torch.inference_mode():
        h1 = F.relu(model.fc_in(Xb))
        h2 = F.relu(model.fc_h1(h1))
        h3 = F.relu(model.fc_h2(h2))
        return [float(h.abs().mean()) for h in (h1, h2, h3, model(Xb))]

def coord_check(arch, widths, seed, lr=2**-8, num_steps=1):
    Xb, Yb = X[:256], Y[:256]
    sizes = []
    for width in widths:
        torch.manual_seed(seed)
        model = arch(width).to(device)
        optimizer = model.configure_adam(lr)
        for _ in range(num_steps):
            train_step(model, optimizer, Xb, Yb)
        sizes.append(activations(model, Xb))
    return list(zip(*sizes))

First under standard parametrization. The check takes seconds, so we extend
the widths to 4,096, beyond those of the training sweep:

In [ ]:
check_widths = [128, 256, 512, 1024, 2048, 4096]
sp_acts = coord_check(MLP, check_widths, seed)
d2l.plot(check_widths, list(sp_acts), 'width', 'mean |activation|',
         xscale='log', yscale='log', ylim=[0.01, 100],
         legend=['layer 1', 'layer 2', 'layer 3', 'logits'])

The plot shows the equation at work. Layer 1, behind the
input matrix with its fixed fan-in, is flat. Layers 2 and 3, behind the
square matrices, grow with width, and the growth compounds with depth
because each layer passes its excess on to the next. The logits grow by
about two orders of magnitude across the range. A first step that is stable
at narrow widths therefore destabilizes a wide enough model. The same check
under muP, on the same axes:

In [ ]:
mup_acts = coord_check(MuMLP, check_widths, seed)
d2l.plot(check_widths, list(mup_acts), 'width', 'mean |activation|',
         xscale='log', yscale='log', ylim=[0.01, 100],
         legend=['layer 1', 'layer 2', 'layer 3', 'logits'])

All three layer curves are flat: no layer's update size depends on the
width anymore. The logits *fall* with width, and this is by design. The
$1/m$ multiplier keeps the logits of the wider networks small at first, and
they grow to order one through learning rather than through size. Growth is the unambiguous failure. A flat
or falling curve says only that the forward pass is stable, because the
check reads the sizes of activations, not whether features are learned.
Updates that shrink too fast with width would also produce falling curves,
with the layer frozen. The test still catches the common implementation
errors: a missing multiplier, a mislabeled layer, or an initializer with a
fixed standard deviation, like the 0.02 common in transformer code, in place
of one that scales with the fan-in. The exercises break each of the two
rules in turn and ask which errors the check detects.

### Learning-Rate Transfer

The coordinate check tests the scaling of activations. Whether the learning
rate now transfers is a question for the sweep, which we repeat under muP
with the same seed:

In [ ]:
mup_loss = {w: [train_mlp(MuMLP, w, lr, seed) for lr in lrs] for w in widths}
for w in widths:
    print(f'width {w:4d}: best lr {min(zip(mup_loss[w], lrs))[1]:.1e}')
d2l.plot(lrs, [mup_loss[w] for w in widths], 'learning rate',
         'training loss', xscale='log', ylim=[0.25, 0.8],
         legend=[f'width {w}' for w in widths])

The width-1,024 curve repeats the earlier one, because at the base width the
two parametrizations coincide and the seed is the same. The narrower curves
have moved to meet it, and the minimum no longer drifts with width: across
the eightfold change in width, each grid minimum lies within one or two grid
steps of the width-128 minimum. To
measure what transfer is worth, we reuse the learning rate selected at width
128 for width 1,024 and compare the result with the best loss of the
width-1,024 sweep. We also print the best loss at each width, the result
of tuning every width separately:

In [ ]:
for name, loss in (('SP', sp_loss), ('muP', mup_loss)):
    tuned = loss[128].index(min(loss[128]))  # Grid index chosen at width 128
    excess = loss[1024][tuned] / min(loss[1024]) - 1
    best = '  '.join(f'{min(loss[w]):.3f}' for w in widths)
    print(f'{name:3s}  best loss at widths {widths}: {best}')
    print(f'     width-128 lr at width 1024: {excess:.0%} above its best')

Under muP the transferred learning rate typically lands within a few
percent of the best loss at width 1,024. Under standard parametrization it
typically lands 10% to 30% above it. Each number comes from a single seed
and varies from run to run, so an occasional run brings the two close
together; the first exercise measures the spread. This is *hyperparameter transfer*: tune the small model,
scale up, keep the numbers. Under muP the best loss also falls from width
128 to width 1,024. Retuned at width 1,024, standard parametrization reaches
the same best loss, because there the two parametrizations coincide. On a
family this small that sweep is still cheap; muP saves it.
@Yang.Hu.Babuschkin.ea.2022 find that the two diverge at larger
widths: the tuned loss under standard parametrization stops improving, while
under muP it keeps falling. Transfer concerns the base learning rate, not
every hyperparameter: schedules, batch size (that section), and
regularization still interact with scale. At a large scale,
@Yang.Hu.Babuschkin.ea.2022 tuned a 6.7-billion-parameter GPT-3 by
sweeping a 40-million-parameter proxy, spent about 7% of the pretraining
compute on tuning, and outperformed the original model.

## The Spectral View

muP arrived through limits of infinite width, but a shorter route connects
it to that section. There we measured the size of a matrix update by
its spectral norm, the largest factor by which it can stretch an activation
vector. For a layer that maps $n_{\textrm{in}}$ activations to
$n_{\textrm{out}}$, with entries of order one on both sides, the healthy
size is a spectral norm of order $\sqrt{n_{\textrm{out}} / n_{\textrm{in}}}$,
for the weights and for their updates alike:

$$
\|\mathbf{W}\|_2 \asymp \sqrt{\frac{n_{\textrm{out}}}{n_{\textrm{in}}}},
\qquad
\|\Delta\mathbf{W}\|_2 \asymp \sqrt{\frac{n_{\textrm{out}}}{n_{\textrm{in}}}}.
$$

An input with entries of order one has length $\sqrt{n_{\textrm{in}}}$, and an
output with entries of order one has length $\sqrt{n_{\textrm{out}}}$, so a
matrix that maps one to the other stretches by the ratio of the two.
@Yang.Simon.Bernstein.2023 show that this *spectral condition*
recovers muP. Adam's first update of a hidden $n \times n$ matrix, the
rank-one sign matrix behind the equation, has a spectral
norm of order $\eta n$, about $\eta n / 2$ when half the ReLU units are active
(an exercise derives it). The condition asks for order one, and dividing the
hidden learning rate by $m$ removes the excess factor $n / n_{\textrm{base}}$.
Seen this way, the per-layer rules of the table bring Adam's
coordinatewise updates to the spectral scale at every width.

An optimizer whose update is already a matrix of controlled spectral norm
needs fewer of these corrections. Muon replaces the update of each hidden
matrix by an orthogonalized one, whose singular values all equal one
[@Jordan.Jin.Boza.ea.2024; @Bernstein.Newhouse.2024]. Multiplied by
$\sqrt{n_{\textrm{out}} / n_{\textrm{in}}}$ [@Bernstein.2025], that
update meets the equation at every width by construction.
The scale used in that section,
$0.2\sqrt{\max(n_{\textrm{out}}, n_{\textrm{in}})}$, was chosen instead to
match the root-mean-square entry of an AdamW update. It gives a square $n \times n$ update a spectral norm of
$0.2\,\eta\sqrt{n}$, which still grows with width, though more slowly than the
$\eta n$ of Adam's first step. Muon at that scale therefore still needs width
corrections for its learning rate to transfer, and
@Shah.Polloreno.Stratos.ea.2025 pair it with muP to transfer
hyperparameters across model sizes.

Width is only one direction of scaling. Transfer across depth, batch size,
and training length needs rules of its own, and whether weight decay rather
than parametrization keeps long runs stable is still debated
[@Kosson.Welborn.Liu.ea.2025]. The coordinate check is worth running on
any family of models one intends to scale, whatever the parametrization.

## Summary

Hyperparameters tuned on a small model do not transfer by default: the best
learning rate falls as the network widens, because one Adam step perturbs a
hidden layer's activations in proportion to its fan-in
(the equation). The maximal update parametrization removes
the width dependence with per-layer rules: the usual initialization with
variance $\propto 1/\textrm{fan-in}$, hidden-matrix learning rates divided by
the width multiplier $m$, logits divided by $m$, and input weights and biases
left alone. These rules fix how each layer scales with the width; the choice
of base width fixes the constant factors. In our family of networks, with
the base at the widest member, the best learning rate then stays within a
grid step or two of its width-128 value across an eightfold change in width.
The coordinate check tests such rules in seconds: after a training step,
activations should follow the width dependence that the parametrization
prescribes, without unintended growth. In spectral terms, muP makes Adam's updates land at the
norm that a layer's shape prescribes, and Muon's orthogonalized updates
carry part of that scaling by construction.

## Exercises

1. [code] **Transfer across seeds.** `sp_loss` and `mup_loss` hold one run
   per width and learning rate, all from one seed, so the transfer cost
   that they show carries no error bar.
    1. Under each parametrization, sweep `lrs` at widths 128 and 1,024 for
       three seeds. For each seed, apply that seed's width-128 optimum at
       width 1,024 and report, in percent, how far its loss lies above the
       minimum of the same seed's width-1,024 sweep.
    1. Is the difference between the two parametrizations' transfer costs
       larger than the spread of each cost across seeds?

1. [code] **Breaking muP on purpose.** The coordinate check is a test for
   implementation errors.
    1. Delete the $1/m$ logit scaling from `MuMLP` but keep the hidden
       learning-rate rule and rerun `coord_check`. Which curve exposes the
       bug?
    1. Restore the multiplier and instead give the hidden matrices the full
       learning rate. Which curve exposes this bug?
    1. Restore the hidden learning-rate rule, but divide the learning rate
       by $m^2$ instead of $m$. Does `coord_check` flag this error? Sweep
       `lrs` at width 1,024 with this variant, and compare its best
       learning rate and best loss with those of `MuMLP` in `mup_loss`.

1. [code] **Choosing the base.** `MuMLP` puts its base at width 1,024, the
   widest member of the family.
    1. Rerun the muP sweep with `base_width=128`, so that muP coincides with
       standard parametrization at the tuned width instead. Does the best
       learning rate still transfer from width 128 to width 1,024? Compare
       the best loss at each width with `mup_loss` and `sp_loss`.
    1. The two bases differ only by constant factors per layer. List them,
       and find which of them accounts for the difference at width 1,024 by
       changing one at a time.

1. [code] **Fitting the drift.** Instead of removing the drift, a large
   project can measure it: @Bi.Chen.Chen.ea.2024 fitted power laws
   for the optimal learning rate and batch size as functions of the compute
   budget across many small runs and extrapolated them to the target scale.
   In the first-step analysis of the equation, the largest
   stable learning rate scales as $1/n$.
    1. From `sp_loss`, fit a power law $\eta^\star(n) = c\, n^{-\gamma}$ to
       the best learning rate at each width. Since `lrs` is spaced by
       factors of two, each optimum is resolved only to within a factor of
       $\sqrt{2}$. Give the range of $\gamma$ consistent with this
       resolution, and state whether it contains $\gamma = 1$.
    1. Extrapolate the fit to widths 2,048 and 4,096. Sweep directly at
       those widths with `train_mlp`, extending `lrs` downward if a
       prediction falls below its smallest value, and report how many grid
       steps separate each prediction from the measured optimum.

1. [code] **Depth and the coordinate check.** ● `coord_check` varies width
   at a fixed depth of three hidden layers. Generalize `MLP` and `MuMLP` to
   $L$ hidden layers and run the check across $L \in \{3, 6, 12, 24\}$ at
   width 512 under both parametrizations. The variance-$1/\textrm{fan-in}$
   initialization of `MLP` halves the second moment of the ReLU
   activations at every layer (that section), so for this
   problem initialize the input and hidden matrices with variance
   $2/\textrm{fan-in}$.
    1. Under standard parametrization, how does the growth of the logits
       with depth compare with their growth with width in
       that section?
    1. Does muP's width scaling keep the activations flat across depth?
       State what the rules of the table promise and what
       they do not.

1. **First update in spectral norm.** ● For a single example, Adam's first
   update of a weight matrix
   $\mathbf{W} \in \mathbb{R}^{n_{\textrm{out}} \times n_{\textrm{in}}}$ is
   $\Delta\mathbf{W} = -\eta\, \operatorname{sign}(\boldsymbol{\delta})\,
   \operatorname{sign}(\mathbf{h})^\top$, as in the derivation
   of the equation. Let $k_{\textrm{out}}$ and
   $k_{\textrm{in}}$ be the numbers of nonzero entries of
   $\boldsymbol{\delta}$ and $\mathbf{h}$.
    1. Compute $\|\Delta\mathbf{W}\|_2$ as a function of $\eta$,
       $k_{\textrm{out}}$, and $k_{\textrm{in}}$.
    1. Assume that $k_{\textrm{out}}$ and $k_{\textrm{in}}$ are fixed
       fractions of $n_{\textrm{out}}$ and $n_{\textrm{in}}$. For the input,
       hidden, and output matrices of `MLP`, find how $\eta$ must scale
       with the width $n$ for $\Delta\mathbf{W}$ to satisfy the spectral
       condition the equation.
    1. Show that the learning rates and the logit multiplier of
       the table give every layer an update at this scale.

1. [code] **Weight decay and parametrization.** ●
   @Kosson.Welborn.Liu.ea.2025 argue that in long runs weight
   decay, not parametrization, stabilizes the update dynamics across
   widths. Override `configure_adam` in a subclass of `MLP` to return
   AdamW (that section) with decay coefficient
   $\lambda = 10^{-3}/\eta$, so that every run averages its updates over
   the horizon $1/(\eta\lambda) = 1{,}000$ steps described
   in that section.
    1. Run the standard-parametrization sweep of
       that section at widths 128 and 1,024 with
       `num_steps=4000`, once with this AdamW and once with Adam. Report
       the best learning rate at each width for both optimizers. Does the
       optimum drift less with width under AdamW?
    1. `MLP` has no normalization layer. Given the equilibrium argument of
       that section, state what your result can and cannot
       say about the cited claim.

[Discussions](https://d2l.discourse.group/t/36766)